In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge

In [2]:
# Data Collection and inital exploration 
data = pd.read_csv("../datasets/house-price/Bangalore.csv")
print(data.head())

      Price  Area                         Location  No. of Bedrooms  Resale  \
0  30000000  3340                 JP Nagar Phase 1                4       0   
1   7888000  1045       Dasarahalli on Tumkur Road                2       0   
2   4866000  1179  Kannur on Thanisandra Main Road                2       0   
3   8358000  1675                     Doddanekundi                3       0   
4   6845000  1670                          Kengeri                3       0   

   MaintenanceStaff  Gymnasium  SwimmingPool  LandscapedGardens  JoggingTrack  \
0                 1          1             1                  1             1   
1                 0          1             1                  1             1   
2                 0          1             1                  1             1   
3                 0          0             0                  0             0   
4                 1          1             1                  1             1   

   ...  LiftAvailable  BED  VaastuComp

In [3]:
# X -> Input features (all columns except SalePrice)
# y -> Target variable (SalePrice)
X = data.drop("Price", axis=1)
y = data["Price"]

# Select numerical columns (int and float types)
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
# Select categorical columns (object/string types)
cat_cols = X.select_dtypes(include=["object"]).columns

# Numerical pipeline:
# - Impute missing values using the median (robust to outliers)
num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

# Categorical pipeline:
# - Impute missing values using most frequent category
# - Encode categories using One-Hot Encoding
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# ==========================================
# Combine Pipelines Using ColumnTransformer
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            num_pipeline,
            num_cols,
        ),  # Apply numerical pipeline to numerical columns
        (
            "cat",
            cat_pipeline,
            cat_cols,
        ),  # Apply categorical pipeline to categorical columns
    ]
)
# =========================
# Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% data for testing
    random_state=42,  # Ensures reproducibility
)


# Create Final Machine Learning Pipeline
lasso_model = Pipeline([("preprocess", preprocessor), ("model", Lasso(alpha=0.001))])

ridge_model = Pipeline([("preprocess", preprocessor), ("model", Ridge(alpha=1.0))])
lasso_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)

print("Lasso R2:", r2_score(y_test, lasso_model.predict(X_test)))
print("Ridge R2:", r2_score(y_test, ridge_model.predict(X_test)))

joblib.dump(lasso_model, "model_lasso.pkl")
joblib.dump(ridge_model, "model_ridge.pkl")

Lasso R2: 0.1813903156389326
Ridge R2: 0.1709568704204354


c:\Devs\anaconda_navigator\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 2.9874925213559885e+17, tolerance: 90206363126995.2
  model = cd_fast.sparse_enet_coordinate_descent(


['model_ridge.pkl']